# Reproduce Laya's typed-decisions result: three losses, three seeds

**Needs a GPU (Kaggle T4 x2) and the typed-decisions training data.** Committed without outputs: it has not been run
yet, so there are no numbers here. The [Laya README](https://github.com/NandhaKishorM/laya) reports 0.36 zero-shot and 0.77 after fine-tuning on typed-decisions; this
notebook checks whether a decisionsmith full fine-tune gets there, and which loss is best:

- `ce`: soft cross-entropy (decisionsmith's default loss)
- `rlcd`: the loss from Laya's own training notebook
- `proper`: cross-entropy plus spherical and ranked-probability scores

Put the typed-decisions JSONL (Laya's notebook format: `{"state", "questions", "gold"}` per line) next to this
notebook as `typed_decisions.jsonl`. The same test split (seeded) is used for every run.

Runs on: Kaggle T4 x2 GPUs, with the typed-decisions data. Not a laptop.

Built on [Laya](https://github.com/NandhaKishorM/laya) (Apache-2.0) by Nandakishor M / Convai Innovations. decisionsmith is an independent project, not affiliated with TypeSafe AI or Convai Innovations.

In [ ]:
!uv pip install --system -q "decisionsmith[laya]"  # Kaggle without uv: !pip install -q uv first
!nvidia-smi -L

In [ ]:
import json
import statistics

import decisionsmith as ds

DATA = "typed_decisions.jsonl"
results = {}
for loss in ["ce", "rlcd", "proper"]:
    for seed in [0, 1, 2]:
        report = ds.finetune(DATA, base="laya", out="runs/%s-%d" % (loss, seed), train="full", loss=loss, seed=seed)
        acc = report.details["finetuned"]["all"]["accuracy"]
        base = report.details["base"]["all"]["accuracy"]
        results.setdefault(loss, []).append(acc)
        print("%-6s seed %d: base %.3f -> fine-tuned %.3f  go=%s" % (loss, seed, base, acc, report.go))

In [ ]:
for loss, accs in results.items():
    print("%-6s mean %.3f  sd %.3f  (%s)" % (loss, statistics.mean(accs), statistics.stdev(accs), accs))
json.dump(results, open("typed_decisions_results.json", "w", encoding="utf-8"), indent=2)

Share the table in a GitHub issue before changing the default loss.